# Merging the csv files

Library import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json
import pandas as pd
from ipywidgets import interact, IntSlider
from IPython.display import display, Markdown, Image
import os
import glob


In [ ]:

# === CONFIG ===
input_folder = "/Users/mariem/Desktop/Project_ADA/newyorker_caption_contest/data"  # your dataset folder
output_file = "/Users/mariem/Desktop/Project_ADA/merged_caption_contest.csv"  # where to save result
chunksize = 100_000  # safely merge in 100k-row chunks
encoding = "utf-8"   # fallback to 'latin1' if errors

# === GATHER FILES ===
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))

# === DETERMINE UNION OF COLUMNS ===
union_cols = set()
for f in csv_files:
    try:
        hdr = pd.read_csv(f, nrows=0, encoding=encoding)
    except Exception:
        hdr = pd.read_csv(f, nrows=0, encoding="latin1")
    union_cols.update(hdr.columns.tolist())

union_cols = sorted(union_cols)
print(f"Detected {len(union_cols)} columns: {union_cols}")

# === PREP OUTPUT ===
if os.path.exists(output_file):
    os.remove(output_file)

# === HELPER FUNCTION ===
def normalize_chunk(df):
    """Fix or standardize columns (optional tweaks)."""
    if 'votes' in df.columns:
        df['votes'] = pd.to_numeric(df['votes'], errors='coerce').fillna(0).astype(int)
    return df

# === MERGE LOOP ===
first_write = True
total_rows = 0

for i, file in enumerate(CSV_DIR, 1):
    print(f"[{i}/{len(CSV_DIR)}] Processing: {os.path.basename(file)}")

    try:
        reader = pd.read_csv(file, chunksize=chunksize, encoding=encoding, low_memory=True)
    except Exception:
        reader = pd.read_csv(file, chunksize=chunksize, encoding="latin1", low_memory=True)

    for chunk in reader:
        for col in union_cols:
            if col not in chunk.columns:
                chunk[col] = pd.NA
        chunk = chunk.reindex(columns=union_cols)
        chunk = normalize_chunk(chunk)
        chunk.to_csv(output_file, mode="a", index=False, header=first_write)
        first_write = False
        total_rows += len(chunk)




Found 384 CSV files.
Detected 8 columns: ['caption', 'funny', 'mean', 'not_funny', 'precision', 'rank', 'somewhat_funny', 'votes']
[1/384] Processing: 510.csv
[2/384] Processing: 511.csv
[3/384] Processing: 512.csv
[4/384] Processing: 513.csv
[5/384] Processing: 514.csv
[6/384] Processing: 515.csv
[7/384] Processing: 516.csv
[8/384] Processing: 517.csv
[9/384] Processing: 518.csv
[10/384] Processing: 519.csv
[11/384] Processing: 520.csv
[12/384] Processing: 521.csv
[13/384] Processing: 522.csv
[14/384] Processing: 523.csv
[15/384] Processing: 524.csv
[16/384] Processing: 526.csv
[17/384] Processing: 527.csv
[18/384] Processing: 528.csv
[19/384] Processing: 529.csv
[20/384] Processing: 530.csv
[21/384] Processing: 531.csv
[22/384] Processing: 532.csv
[23/384] Processing: 533.csv
[24/384] Processing: 534.csv
[25/384] Processing: 535.csv
[26/384] Processing: 536.csv
[27/384] Processing: 537.csv
[28/384] Processing: 538.csv
[29/384] Processing: 539.csv
[30/384] Processing: 541.csv
[31/384]

In [ ]:
import pandas as pd
import os

# === CONFIG (edit if needed) ===
in_path  = "/Users/mariem/Desktop/Project_ADA/merged_caption_contest.csv"
out_path = "/Users/mariem/Desktop/Project_ADA/merged_caption_contest_with_contest.csv"
start_contest = 510

# === READ ===
df = pd.read_csv(in_path, dtype=str)  # read as str to avoid surprises, we'll coerce rank
print("Rows read:", len(df))

# === CLEAN / COERCE rank ===
# Convert rank to numeric where possible. Non-convertible -> NaN -> set to -1 (not a marker)
df['rank_numeric'] = pd.to_numeric(df.get('rank', None), errors='coerce')
# treat NaN as not-equal-to-zero marker
# We'll consider exactly 0.0 as marker. Use round to avoid floats like 0.000.
df['rank_is_zero'] = df['rank_numeric'].fillna(-9999).apply(lambda x: x == 0)

# === ASSIGN contest numbers ===
contest_num = start_contest - 1
contest_numbers = []
for is_zero in df['rank_is_zero'].tolist():
    if is_zero:
        contest_num += 1
    contest_numbers.append(contest_num)

df['contest_number'] = contest_numbers

# === DIAGNOSTICS ===
num_zero_markers = df['rank_is_zero'].sum()
unique_contests = df['contest_number'].nunique()
min_contest = df['contest_number'].min()
max_contest = df['contest_number'].max()

print(f"Detected rank==0 markers: {int(num_zero_markers)}")
print(f"Unique contest numbers assigned: {unique_contests}")
print(f"Contest number range: {min_contest} .. {max_contest}")

# If you expect exactly 895 contests, check:
expected = 895
if unique_contests == expected:
    print(f"✅ Matches expected contest count: {expected}")
else:
    print(f"⚠️ Does NOT match expected ({expected}).")
    print("Possible reasons: ordering, missing rank==0 markers, or rank values use a different marker.")
    # show first 5 positions where rank==0 (for debugging)
    print("Sample rows where rank==0 (first 10 indices):")
    print(df[df['rank_is_zero']].head(10))

# === SAVE (drop helper cols) ===
df_out = df.drop(columns=['rank_numeric', 'rank_is_zero'])
df_out.to_csv(out_path, index=False)
print("Saved augmented file to:", out_path)


Rows read: 2292678
Detected rank==0 markers: 148
Unique contest numbers assigned: 148
Contest number range: 510 .. 657
⚠️ Does NOT match expected (895).
Possible reasons: ordering, missing rank==0 markers, or rank values use a different marker.
Sample rows where rank==0 (first 10 indices):
                                                 caption funny  \
0              I'm a congressman--obstruction is my job.    18   
3905                                         I'm hourly.    12   
7230                We're pretentious, not ostentatious.    19   
11629  I'm just saying, I can see why the 'brief'case...     3   
15770                                    No, you grow up    53   
19721         Tell me about your childhood very quickly.     7   
23991  I have no concrete plans for the rest of the day.     9   
29909  Who'd have thought they'd get you for tax evas...    14   
37691  This saddest part is he's going in the wrong d...    47   
42200  He's pro-gun, but I like his stance on mig

In [ ]:
import pandas as pd
from pathlib import Path

def check_votes_all_contests(json_df, csv_folder):
    """
    Goes through all contests in json_df ,
    computes total votes from corresponding CSVs, and compares with num_votes
    in metadata (if available).
    
    Returns a DataFrame with the results.
    """
    folder = Path(csv_folder)
    results = []

    for _, row in json_df.iterrows():
        contest_id = str(row["contest_id"])
        metadata = row.get("metadata", {})

        # ensure metadata is a dict (in case it's NaN or string)
        if not isinstance(metadata, dict):
            metadata = {}

        csv_path = folder / f"{contest_id}.csv"

        if not csv_path.exists():
            results.append({
                "contest_id": contest_id,
                "csv_found": False,
                "csv_votes": None,
                "json_votes": metadata.get("num_votes", None),
                "match": None,
                "error": "CSV not found"
            })
            continue

        try:
            df = pd.read_csv(csv_path)

            if "votes" not in df.columns:
                results.append({
                    "contest_id": contest_id,
                    "csv_found": True,
                    "csv_votes": None,
                    "json_votes": metadata.get("num_votes", None),
                    "match": None,
                    "error": "No 'votes' column in CSV"
                })
                continue

            # compute total votes
            total_votes = pd.to_numeric(df["votes"], errors="coerce").sum()
            json_votes = metadata.get("num_votes", None)

            # compare (only if json_votes exists)
            match = (json_votes is not None) and (int(total_votes) == int(json_votes))

            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "csv_votes": int(total_votes),
                "json_votes": json_votes,
                "match": match
            })

        except Exception as e:
            results.append({
                "contest_id": contest_id,
                "csv_found": True,
                "csv_votes": None,
                "json_votes": metadata.get("num_votes", None),
                "match": None,
                "error": str(e)
            })

    df_results = pd.DataFrame(results)

    # Optional summary print
    total = len(df_results)
    matches = df_results["match"].sum(skipna=True)
    print(f"\n=== Sanity Check Summary ===")
    print(f"Total contests checked: {total}")
    print(f"Matching vote counts: {int(matches)}")
    print(f"Mismatches or missing: {total - int(matches)}")
    print("============================\n")

    return df_results
